# Course 1 — Classification + KNN

What does a classifier output? How do we measure it? KNN is the
simplest learner and the perfect place to meet decision boundaries,
confusion matrices, and ROC curves.

**Sections**
1. What a classifier outputs (0:00–0:30)
2. KNN — the no-training baseline (0:30–1:00)
3. Class imbalance and the majority-class trap (1:00–1:30)

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (confusion_matrix, classification_report,
                              roc_curve, auc, ConfusionMatrixDisplay)
rng = np.random.default_rng(0)
default = load_dataset('default')
print(default.head())
print('default rate:', (default['default'] == 'Yes').mean().round(4))


## 1. What a classifier outputs

### Look at the data first

In [ ]:
d = default.copy()
d['y'] = (d['default'] == 'Yes').astype(int)
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(d.loc[d.y==0, 'balance'], d.loc[d.y==0, 'income'], s=3, alpha=0.3, label='No')
ax.scatter(d.loc[d.y==1, 'balance'], d.loc[d.y==1, 'income'], s=10, color='C3', label='Yes')
ax.set_xlabel('balance'); ax.set_ylabel('income'); ax.legend()
ax.set_title('Default — balance is the separating feature'); plt.show()


### Confusion matrix and metrics

The simplest possible classifier: 'predict default if balance > 1700'.
Score it like any other model.

In [ ]:
y = d['y'].to_numpy()
y_hat = (d['balance'] > 1700).astype(int).to_numpy()
cm = confusion_matrix(y, y_hat)
print('confusion matrix [TN FP / FN TP]:'); print(cm)
print()
print(classification_report(y, y_hat, target_names=['No', 'Yes']))


**Precision** = TP / (TP + FP) — of those we flagged, how many really default.
**Recall** = TP / (TP + FN) — of those who really default, how many we flagged.
**F1** = harmonic mean of the two. **AUC** = area under the ROC curve, the
probability that a random positive scores higher than a random negative.

### ROC and AUC

In [ ]:
from sklearn.linear_model import LogisticRegression
X = d[['balance', 'income']].to_numpy()
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
lr = LogisticRegression(max_iter=2000).fit(Xtr, ytr)
scores = lr.predict_proba(Xte)[:, 1]
fpr, tpr, _ = roc_curve(yte, scores)
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(fpr, tpr); ax.plot([0, 1], [0, 1], 'k--', alpha=0.4)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title(f'ROC — AUC = {auc(fpr, tpr):.4f}'); plt.show()


## 2. KNN — the no-training baseline

### Why scale before KNN

In [ ]:
print('balance range:', d['balance'].min(), '...', d['balance'].max())
print('income range :', d['income'].min(), '...', d['income'].max())


Income dwarfs balance numerically — without scaling, KNN's 'nearest'
is decided by income alone.

In [ ]:
pipe = Pipeline([('s', StandardScaler()), ('knn', KNeighborsClassifier(n_neighbors=5))])
pipe.fit(Xtr, ytr)
print(f'KNN(k=5) test accuracy = {pipe.score(Xte, yte):.4f}')


### How does k change the decision boundary?

In [ ]:
# Subsample for plotting speed
rs = rng.choice(len(Xtr), 1500, replace=False)
Xs, ys = Xtr[rs], ytr[rs]
x_min, x_max = Xs[:, 0].min() - 100, Xs[:, 0].max() + 100
y_min, y_max = Xs[:, 1].min() - 1000, Xs[:, 1].max() + 1000
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 120), np.linspace(y_min, y_max, 120))
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for ax, k in zip(axes, [1, 5, 25]):
    pipe = Pipeline([('s', StandardScaler()), ('knn', KNeighborsClassifier(k))])
    pipe.fit(Xs, ys)
    Z = pipe.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.2, levels=[-0.5, 0.5, 1.5])
    ax.scatter(Xs[ys==0, 0], Xs[ys==0, 1], s=3, alpha=0.3)
    ax.scatter(Xs[ys==1, 0], Xs[ys==1, 1], s=12, color='C3')
    ax.set_title(f'k = {k}')
plt.tight_layout(); plt.show()


k = 1 is wild (one outlier carves a region); k = 25 is smooth. Pick k
with cross-validation, not by eye.

## 3. Class imbalance

### The majority-class trap

In [ ]:
trivial = np.zeros_like(yte)  # 'always predict No'
print(f'"always No" accuracy = {(trivial == yte).mean():.4f}')
print('… but recall on defaulters = 0. Useless model.')


### Fix: stratify + class_weight

In [ ]:
from sklearn.linear_model import LogisticRegression
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
lr = LogisticRegression(max_iter=2000, class_weight='balanced').fit(Xtr, ytr)
y_pred = lr.predict(Xte)
print(classification_report(yte, y_pred, target_names=['No', 'Yes']))


Accuracy *drops* but recall on the positive class jumps — almost
always the right trade-off for fraud, default, disease, etc.

### Recap
- Classifier output: a label, often via a probability + threshold.
- Read precision, recall, F1, AUC together. Accuracy alone lies on imbalanced data.
- KNN needs scaling. k controls the bias-variance dial.
- Use `stratify=y` and `class_weight='balanced'` when classes are skewed.

---

# Exercises

Each exercise below is followed by its solution.
Try to solve the tasks yourself before revealing the next cell.

---

## Exercise 1

**Task 1.** Load `penguins`, drop NaN, and fit KNN with `k=5` to
predict `species` from `flipper_length_mm` and `bill_length_mm`.
Use a stratified train/test split and report test accuracy.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
# your code here


### Exercise 1 — Solution

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
df = load_dataset('penguins').dropna()
X = df[['flipper_length_mm', 'bill_length_mm']].to_numpy()
y = df['species'].to_numpy()
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
pipe = Pipeline([('s', StandardScaler()), ('knn', KNeighborsClassifier(5))]).fit(Xtr, ytr)
print(f'k=5 test accuracy = {pipe.score(Xte, yte):.4f}')


---

## Exercise 2

**Task 2.** Plot the decision boundary for `k ∈ {1, 5, 25}` on the
(flipper_length_mm, bill_length_mm) plane.

In [ ]:
# your code here


### Exercise 2 — Solution

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder().fit(y)
y_enc = le.transform(y)
x_min, x_max = X[:,0].min()-2, X[:,0].max()+2
y_min, y_max = X[:,1].min()-2, X[:,1].max()+2
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 120), np.linspace(y_min, y_max, 120))
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for ax, k in zip(axes, [1, 5, 25]):
    pipe = Pipeline([('s', StandardScaler()), ('knn', KNeighborsClassifier(k))]).fit(X, y_enc)
    Z = pipe.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, levels=[-0.5,0.5,1.5,2.5])
    for cls in range(3):
        ax.scatter(X[y_enc==cls,0], X[y_enc==cls,1], s=10, label=le.classes_[cls])
    ax.set_title(f'k = {k}'); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()


---

## Exercise 3

**Task 3.** Use 5-fold CV to pick the best `k` from `[1, 3, 5, 11, 21, 51]`.

In [ ]:
# your code here


### Exercise 3 — Solution

In [ ]:
from sklearn.model_selection import cross_val_score
ks = [1, 3, 5, 11, 21, 51]
for k in ks:
    pipe = Pipeline([('s', StandardScaler()), ('knn', KNeighborsClassifier(k))])
    score = cross_val_score(pipe, X, y, cv=5).mean()
    print(f'k={k:3d}  CV accuracy = {score:.4f}')
